#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col

#Reading data from Bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

###Data transformations

####trim firstname and lastname

In [0]:
#trim the names
#normalization for gndr,marital_status
#headers normalize


for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
      df = df.withColumn(field.name, trim(col(field.name)))

####normalize gender and marital status

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S","Single")
        .when(F.upper(F.col("cst_marital_status")) == "M","Married")
        .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "M","Male")
        .when(F.upper(F.col("cst_gndr")) == "F","Female")
        .otherwise("n/a")
    )
)
         

####Renaming column names

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "customer_firstname",
    "cst_lastname": "customer_lastname",
    "cst_marital_status": "customer_marital_status",
    "cst_gndr": "customer_gender",
    "cst_create_date": "customer_create_date",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)


###Write into silver layer

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("workspace.silver.cust_info")
)